In [1]:
import requests
def distancia_osrm(origen, destino, local_server_url=None):
    """Calcula la distancia en km usando OSRM entre dos puntos (lat, lon). Usa servidor local si se indica, si no el público. Retorna None si falla."""
    base_url = local_server_url if local_server_url else "http://router.project-osrm.org"
    url = f"{base_url}/route/v1/driving/{origen[1]},{origen[0]};{destino[1]},{destino[0]}?overview=false"
    try:
        response = requests.get(url, timeout=5)  # timeout de 5 segundos
        if response.status_code != 200:
            print(f"Error OSRM: status {response.status_code} - {response.text}")
            return None
        data = response.json()
        if 'routes' in data and len(data['routes']) > 0:
            distancia_m = data['routes'][0]['distance']
            return distancia_m / 1000  # Convertir a km
        else:
            print(f"Error OSRM: respuesta sin rutas - {data}")
            return None
    except requests.exceptions.Timeout:
        print("Error OSRM: timeout excedido")
        return None
    except Exception as e:
        print(f"Error OSRM: {e}")
        return None

In [2]:
# ================= LIBRERÍAS, FUNCIONES Y EXPORTACIÓN =================
import pandas as pd, numpy as np, geopandas as gpd, matplotlib.pyplot as plt, seaborn as sns, folium, os, time, warnings
from shapely.geometry import Point
from folium.plugins import MarkerCluster
from datetime import datetime
from geopy.distance import geodesic
import openpyxl
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Font, PatternFill, Alignment
from io import BytesIO
warnings.filterwarnings('ignore')
plt.style.use('default'); sns.set_palette('husl')

def cargar_datos(archivo):
    try:
        df_cli = pd.read_excel(archivo, sheet_name='Clientes').rename(columns={'U_latitud':'Latitud','U_longitud':'Longitud','CodCli':'Codigo_Cliente'}).dropna()
        df_cdr = pd.read_excel(archivo, sheet_name='CDR').rename(columns={'U_latitud':'Latitud','U_longitud':'Longitud','CodMET':'CDR'})
        cdr_nombres = {df_cdr.iloc[0]['CDR']: 'MET Celaya', df_cdr.iloc[1]['CDR']: 'MET Queretaro'}
        df_cdr['CDR'] = df_cdr['CDR'].map(cdr_nombres)
        gdf_cli = gpd.GeoDataFrame(df_cli, geometry=[Point(lon, lat) for lon, lat in zip(df_cli['Longitud'], df_cli['Latitud'])], crs='EPSG:4326')
        gdf_cdr = gpd.GeoDataFrame(df_cdr, geometry=[Point(lon, lat) for lon, lat in zip(df_cdr['Longitud'], df_cdr['Latitud'])], crs='EPSG:4326')
        return gdf_cli, gdf_cdr
    except Exception as e:
        print(f'Error cargando datos: {e}')
        return None, None

def analizar_distancias(gdf_cli, gdf_cdr, cantidad=None, batch=1000, tolerancia=2):
    if cantidad and cantidad < len(gdf_cli):
        gdf_cli = gdf_cli.sample(n=cantidad, random_state=42)
    celaya = (gdf_cdr[gdf_cdr['CDR']=='MET Celaya']['Latitud'].iloc[0], gdf_cdr[gdf_cdr['CDR']=='MET Celaya']['Longitud'].iloc[0])
    queretaro = (gdf_cdr[gdf_cdr['CDR']=='MET Queretaro']['Latitud'].iloc[0], gdf_cdr[gdf_cdr['CDR']=='MET Queretaro']['Longitud'].iloc[0])
    resultados = []
    for idx, cli in gdf_cli.iterrows():
        dist_celaya = geodesic((cli['Latitud'], cli['Longitud']), celaya).kilometers
        dist_queretaro = geodesic((cli['Latitud'], cli['Longitud']), queretaro).kilometers
        dif = dist_celaya - dist_queretaro
        if abs(dif) <= tolerancia:
            mejor = 'Se queda igual'; cat = 'Neutral'; ahorro = 0
        elif dif > 0:
            mejor = 'MET Queretaro'; cat = 'Beneficia Queretaro'; ahorro = dif
        else:
            mejor = 'MET Celaya'; cat = 'Beneficia Celaya'; ahorro = abs(dif)
        resultados.append({'Cliente': cli['Codigo_Cliente'], 'Latitud': cli['Latitud'], 'Longitud': cli['Longitud'], 'Dist_Celaya_km': round(dist_celaya,2), 'Dist_Queretaro_km': round(dist_queretaro,2), 'Diferencia_km': round(dif,2), 'Mejor_Opcion': mejor, 'Ahorro_km': round(ahorro,2), 'Categoria': cat})
    return pd.DataFrame(resultados)

def crear_mapa_cluster(df_analisis, gdf_cdr, directorio, timestamp):
    centro_lat = df_analisis['Latitud'].mean()
    centro_lon = df_analisis['Longitud'].mean()
    mapa = folium.Map(location=[centro_lat, centro_lon], zoom_start=8, tiles='OpenStreetMap')
    cluster = MarkerCluster().add_to(mapa)
    # Popups de clientes: formato horizontal y ancho
    for idx, row in df_analisis.iterrows():
        popup_html = f'''
        <div style="width:320px; padding:10px 12px; background:#fff; border-radius:10px; box-shadow:0 2px 8px #aaa; font-size:15px;">
            <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
                <span style="font-weight:bold; color:#00529B; font-size:17px;">Cliente: {row['Cliente']}</span>
                <span style="font-size:22px;">🧑‍💼</span>
            </div>
            <div style="display:flex; justify-content:space-between;">
                <span>📚 Celaya:<b> {row['Dist_Celaya_km']} km</b></span>
                <span>📚 Querétaro:<b> {row['Dist_Queretaro_km']} km</b></span>
            </div>
            <div style="display:flex; justify-content:space-between; margin-top:6px;">
                <span>🎯 Mejor:<b> {row['Mejor_Opcion']}</b></span>
                <span>💡 Ahorro:<b> {row['Ahorro_km']} km</b></span>
            </div>
        </div>'''
        folium.Marker(location=[row['Latitud'], row['Longitud']], popup=popup_html).add_to(cluster)
    # Icono personalizado para los CDR
    icon_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
    cdr_icon = folium.CustomIcon(icon_image=icon_path, icon_size=(38,38))
    # Popups de MEET: color, icono y texto destacado
    for idx, cdr in gdf_cdr.iterrows():
        meet_color = '#FF6B00' if cdr['CDR']=='MET Celaya' else '#00529B'
        meet_icon = '🏢' if cdr['CDR']=='MET Celaya' else '🏢'
        popup_html = f'''
        <div style="width:220px; padding:12px; background:{meet_color}; color:#fff; border-radius:10px; box-shadow:0 2px 8px #aaa; text-align:center; font-size:16px;">
            <div style="font-size:32px;">{meet_icon}</div>
            <div style="font-weight:bold; font-size:18px; margin-top:4px;">{cdr['CDR']}</div>
            <div style="font-size:14px; margin-top:2px;">Lat: {cdr['Latitud']}<br>Lon: {cdr['Longitud']}</div>
        </div>'''
        folium.Marker(location=[cdr['Latitud'], cdr['Longitud']], popup=popup_html, icon=cdr_icon).add_to(mapa)
    # Título principal arriba del mapa
    titulo_html = '''
    <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9999; background-color: white; padding: 10px 30px; border: 2px solid #333; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.3);">
        <h2 style="margin: 0; color: #333; text-align: center;">
            <span style='font-size:22px;'>🗺️ MAPA DE CONVENIENCIA - ANÁLISIS CDR</span>
        </h2>
        <p style="margin: 5px 0 0 0; text-align: center; color: #666; font-size: 16px;">
            Celaya vs Querétaro | {} Clientes Analizados
        </p>
    </div>
    '''.format(len(df_analisis))
    mapa.get_root().html.add_child(folium.Element(titulo_html))
    # Panel resumen a la derecha
    resumen_html = '''
    <div style="position: fixed; top: 20px; right: 20px; width: 340px; background-color: white; border: 2px solid #333; z-index: 9999; font-size: 15px; padding: 18px; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.3);">
        <h3 style="margin-top: 0; color: #333; text-align: left; border-bottom: 2px solid #ddd; padding-bottom: 8px;">
            <span style='font-size:20px;'>📊 ANÁLISIS CDR - CONVENIENCIA</span>
        </h3>
        <p style="margin: 8px 0; font-weight: bold; color: #333;">
            <span style='font-size:17px;'>🧮 Total clientes: {}</span>
        </p>
        <h4 style="margin: 12px 0 8px 0; color: #444;">🎯 DISTRIBUCIÓN POR CDR:</h4>
        <p style="margin: 4px 0;"><span style="color: #FF0000; font-size: 16px;">●</span> <strong>MET Celaya:</strong> {} clientes ({}%)</p>
        <p style="margin: 4px 0;"><span style="color: #0000FF; font-size: 16px;">●</span> <strong>MET Querétaro:</strong> {} clientes ({}%)</p>
        <p style="margin: 4px 0;"><span style="color: #FFA500; font-size: 16px;">●</span> <strong>Neutral:</strong> {} clientes ({}%)</p>
    </div>
    '''.format(
        len(df_analisis),
        len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya']), round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya'])/len(df_analisis)*100,1),
        len(df_analisis[df_analisis['Mejor_Opcion']=='MET Queretaro']), round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Queretaro'])/len(df_analisis)*100,1),
        len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual']), round(len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual'])/len(df_analisis)*100,1)
    )
    mapa.get_root().html.add_child(folium.Element(resumen_html))
    archivo_mapa = os.path.join(directorio, f'Mapa_Cluster_{timestamp}.html')
    mapa.save(archivo_mapa)
    return archivo_mapa

In [3]:
def exportar_excel(df_analisis, directorio, timestamp, cantidad, grafica_buf):
    archivo_excel = os.path.join(directorio, f'Analisis_CDR_Adaptable_{len(df_analisis)}_Clientes_{timestamp}.xlsx')
    with pd.ExcelWriter(archivo_excel, engine='openpyxl') as writer:
        # Hoja 1: Resumen Ejecutivo (más completo y con emojis)
        resumen = [
            ['📊 MÉTRICAS PRINCIPALES', ''],
            ['👥 Total clientes analizados', len(df_analisis)],
            ['🏢 MET Celaya', len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya'])],
            ['🏢 MET Querétaro', len(df_analisis[df_analisis['Mejor_Opcion']=='MET Queretaro'])],
            ['⚖️ Neutrales', len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual'])],
            ['💰 Ahorro total (km)', round(df_analisis['Ahorro_km'].sum(),2)],
            ['📈 Ahorro promedio (km)', round(df_analisis['Ahorro_km'].mean(),2)],
            ['🏆 Máximo ahorro individual (km)', round(df_analisis['Ahorro_km'].max(),2)],
            ['🔢 Mediana ahorro (km)', round(df_analisis['Ahorro_km'].median(),2)],
            ['📊 % Celaya', round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya'])/len(df_analisis)*100,1)],
            ['📊 % Querétaro', round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Queretaro'])/len(df_analisis)*100,1)],
            ['📊 % Neutral', round(len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual'])/len(df_analisis)*100,1)],
            ['🗓️ Fecha análisis', timestamp],
            ['📄 Archivo fuente', os.path.basename(directorio)],
        ]
        df_resumen = pd.DataFrame(resumen, columns=['Métrica','Valor'])
        df_resumen.to_excel(writer, sheet_name='Resumen_Ejecutivo', index=False)
        # Hoja 2: Detalle
        df_analisis.to_excel(writer, sheet_name='Analisis_Detallado', index=False)
    # Insertar gráfica en Excel
    wb = openpyxl.load_workbook(archivo_excel)
    ws = wb['Resumen_Ejecutivo']
    img = XLImage(grafica_buf)
    img.width = 420
    img.height = 220
    img.anchor = 'C2'
    ws.add_image(img)
    # Formato: títulos y colores (naranja oscuro)
    naranja_oscuro = 'C25A00'
    for row in ws.iter_rows(min_row=2, max_row=2, min_col=1, max_col=2):
        for cell in row:
            cell.font = Font(bold=True, color='FFFFFF', size=14)
            cell.fill = PatternFill(start_color=naranja_oscuro, end_color=naranja_oscuro, fill_type='solid')
            cell.alignment = Alignment(horizontal='center')
    for row in ws.iter_rows(min_row=3, max_row=ws.max_row, min_col=1, max_col=2):
        for cell in row:
            cell.font = Font(size=12)
            cell.alignment = Alignment(horizontal='left')
            if row[0].value and 'Celaya' in str(row[0].value):
                cell.fill = PatternFill(start_color='FFCCCC', end_color='FFCCCC', fill_type='solid')
            elif row[0].value and 'Querétaro' in str(row[0].value):
                cell.fill = PatternFill(start_color='CCE5FF', end_color='CCE5FF', fill_type='solid')
            elif row[0].value and 'Neutral' in str(row[0].value):
                cell.fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
            elif row[0].value and 'Ahorro' in str(row[0].value):
                cell.fill = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')
    ws.column_dimensions['A'].width = 32
    ws.column_dimensions['B'].width = 22
    wb.save(archivo_excel)
    return archivo_excel

In [4]:
def crear_visualizaciones(df_analisis):
    buf = BytesIO()
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    # 1. Gráfica de pastel de asignación
    colores = ['#FF6B00', '#00529B', '#FFA500']
    asignacion = df_analisis['Mejor_Opcion'].value_counts()
    axs[0].pie(asignacion.values, labels=asignacion.index, autopct='%1.1f%%', colors=colores[:len(asignacion)], startangle=90)
    axs[0].set_title('Distribución de Asignación de Clientes', fontsize=13, color='#FF6B00')
    # 2. Histogramas de distancia para cada MEET
    axs[1].hist(df_analisis['Dist_Celaya_km'], bins=25, alpha=0.7, label='MET Celaya', color='#FF6B00', edgecolor='black')
    axs[1].hist(df_analisis['Dist_Queretaro_km'], bins=25, alpha=0.7, label='MET Querétaro', color='#00529B', edgecolor='black')
    axs[1].set_xlabel('Distancia (km)', fontsize=11)
    axs[1].set_ylabel('Frecuencia', fontsize=11)
    axs[1].set_title('Histograma de Distancias por MEET', fontsize=13, color='#FF6B00')
    axs[1].legend(fontsize=11)
    axs[1].grid(True, alpha=0.3)
    # 3. Dispersión de clientes y distancias por MEET
    colores_disp = {'MET Celaya': '#FF6B00', 'MET Queretaro': '#00529B', 'Se queda igual': '#FFA500'}
    for meet, color in colores_disp.items():
        subset = df_analisis[df_analisis['Mejor_Opcion'] == meet]
        axs[2].scatter(subset['Longitud'], subset['Latitud'], s=20, alpha=0.7, c=color, label=meet)
    axs[2].set_xlabel('Longitud', fontsize=11)
    axs[2].set_ylabel('Latitud', fontsize=11)
    axs[2].set_title('Dispersión de Clientes por MEET', fontsize=13, color='#FF6B00')
    axs[2].legend(fontsize=11)
    plt.tight_layout()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf

In [5]:
# ================= VARIABLES DE CONFIGURACIÓN =================
ARCHIVO_DATOS = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Celaya_pi.xlsx'
DIRECTORIO_RESULTADOS = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Resultados'
CANTIDAD_CLIENTES_ANALIZAR = 4415
BATCH_SIZE = 4415
TOLERANCIA_NEUTRAL_KM = 2
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs(DIRECTORIO_RESULTADOS, exist_ok=True)

In [6]:
def analizar_distancias(gdf_cli, gdf_cdr, cantidad=None, batch=1000, tolerancia=2, local_server_url=None):
    if cantidad and cantidad < len(gdf_cli):
        gdf_cli = gdf_cli.sample(n=cantidad, random_state=42)
    celaya = (gdf_cdr[gdf_cdr['CDR']=='MET Celaya']['Latitud'].iloc[0], gdf_cdr[gdf_cdr['CDR']=='MET Celaya']['Longitud'].iloc[0])
    queretaro = (gdf_cdr[gdf_cdr['CDR']=='MET Queretaro']['Latitud'].iloc[0], gdf_cdr[gdf_cdr['CDR']=='MET Queretaro']['Longitud'].iloc[0])
    resultados = []
    for idx, cli in gdf_cli.iterrows():
        # Usar OSRM para distancias reales por carretera
        dist_celaya = distancia_osrm((cli['Latitud'], cli['Longitud']), celaya, local_server_url=local_server_url)
        dist_queretaro = distancia_osrm((cli['Latitud'], cli['Longitud']), queretaro, local_server_url=local_server_url)
        # Si OSRM falla, usar geodesic como respaldo
        if dist_celaya is None:
            dist_celaya = geodesic((cli['Latitud'], cli['Longitud']), celaya).kilometers
        if dist_queretaro is None:
            dist_queretaro = geodesic((cli['Latitud'], cli['Longitud']), queretaro).kilometers
        dif = dist_celaya - dist_queretaro
        if abs(dif) <= tolerancia:
            mejor = 'Se queda igual'; cat = 'Neutral'; ahorro = 0
        elif dif > 0:
            mejor = 'MET Queretaro'; cat = 'Beneficia Queretaro'; ahorro = dif
        else:
            mejor = 'MET Celaya'; cat = 'Beneficia Celaya'; ahorro = abs(dif)
        resultados.append({'Cliente': cli['Codigo_Cliente'], 'Latitud': cli['Latitud'], 'Longitud': cli['Longitud'], 'Dist_Celaya_km': round(dist_celaya,2), 'Dist_Queretaro_km': round(dist_queretaro,2), 'Diferencia_km': round(dif,2), 'Mejor_Opcion': mejor, 'Ahorro_km': round(ahorro,2), 'Categoria': cat})
    return pd.DataFrame(resultados)


In [ ]:
# ================= MACROPROCESOS =================
# 1. Carga de datos
gdf_clientes, gdf_cdr = cargar_datos(ARCHIVO_DATOS)
if gdf_clientes is None or gdf_cdr is None:
    raise Exception('No se pudieron cargar los datos')

# 2. Análisis de distancias
# Cambia la URL aquí si tienes OSRM local:
OSRM_LOCAL_URL = 'http://localhost:5000'  # Usar servidor local
df_analisis = analizar_distancias(gdf_clientes, gdf_cdr, cantidad=CANTIDAD_CLIENTES_ANALIZAR, batch=BATCH_SIZE, tolerancia=TOLERANCIA_NEUTRAL_KM, local_server_url=OSRM_LOCAL_URL)

# 3. Mapa con clustering
archivo_mapa = crear_mapa_cluster(df_analisis, gdf_cdr, DIRECTORIO_RESULTADOS, timestamp)

# 4. Visualización y exportación
grafica_buf = crear_visualizaciones(df_analisis)
archivo_excel = exportar_excel(df_analisis, DIRECTORIO_RESULTADOS, timestamp, CANTIDAD_CLIENTES_ANALIZAR, grafica_buf)

# Mostrar el mapa y la ruta del archivo Excel generado
from IPython.display import display, IFrame

display(IFrame(src=archivo_mapa, width=900, height=600))
print('Reporte Excel generado en:', archivo_excel)


# ? Análisis de Distancias CDR: Celaya vs Querétaro

Este notebook realiza un análisis comparativo de distancias entre clientes y dos centros de distribución (CDR):
- **MET Celaya**
- **MET Querétaro**

## 📋 Funcionalidades Específicas:
1. **Lectura de datos**: Importar desde Excel con hojas Clientes y CDR
2. **Análisis de distancias**: Cálculo de distancias reales por carretera (no lineales)
3. **Comparación CDR**: Determinación del CDR más cercano para cada cliente
4. **Optimización de rutas**: Análisis de eficiencia logística
5. **Visualización**: Mapas interactivos y gráficos comparativos
6. **Reporte final**: Exportación a Excel con recomendaciones

## 📂 Archivos de trabajo:
- **Entrada**: `Celaya_pi.xlsx` (hojas: Clientes, CDR)
- **Salida**: Reporte completo en carpeta `Resultados`